# CASMI 2026 — PubChem-wide candidates + fingerprint model + library cosine (v3)

Extends the library-retrieval baseline to **classes 2/3-ish** (structure known to PubChem/COCONUT, no public spectra):

0. **v3 change**: besides the pool below, candidates also come from a ±10 ppm window of the full **105.9M-structure PubChem tier** (top by PubChem popularity), so molecules outside COCONUT/training can be found.
1. **Candidate pool** = `pool_popularity.csv` (710k training + COCONUT structures with PubChem popularity) ∪ natural-product knowledge table (LOTUS/NPAtlas/COCONUT, 152k).
2. Per test molecule: shortlist candidates by neutral mass.
3. Score each candidate with
   * `lib` — best-match spectral cosine against library spectra (0 if the structure has no library spectra),
   * `fp` — expected Tanimoto between the fingerprint predicted from the molecule's spectra (`fpmodel.pt`, trained in `casmi26-fp-model-train`) and the candidate's Morgan fingerprint, z-scored within the shortlist,
   * `pop` — PubChem popularity prior (`log1p(substances)+log1p(pubmed)`),
   * `np` — natural-product flag.
4. Blend with weights tuned on a local validation mix (class-1-like and held-out-structure class-2-like molecules), output top 25.

Data used: competition data, `dmitriigluzdov/casmi26-pubchem-popularity-prior`, `dmitriigluzdov/casmi26-natural-product-knowledge-table`, and kernel output of `dalloliogm/casmi26-fp-model-train`.

In [ ]:
import sys, subprocess, importlib.metadata, glob
try: _v = importlib.metadata.version("rdkit")
except importlib.metadata.PackageNotFoundError: _v = ""
if not _v:
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    w = sorted(glob.glob(f"/kaggle/input/**/rdkit-*-{tag}-*.whl", recursive=True))
    assert w, f"attach an offline RDKit wheel for {tag}"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-q", w[-1]])


In [ ]:
import os, re, time, glob, itertools
import numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow.compute as pc, pyarrow as pa
import scipy.sparse as sp, torch, torch.nn as nn
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")
def find(name, env):
    p = os.environ.get(env) or next(iter(glob.glob(f"/kaggle/input/**/{name}", recursive=True)), None)
    assert p, f"{name} not found"; return p
TRAIN = find("train.parquet", "CASMI_TRAIN"); DATA_DIR = os.path.dirname(TRAIN)
POOL_CSV, NP_PQ, MODEL_PT = find("pool_popularity.csv", "POOL_CSV"), find("natural_products.parquet", "NP_PQ"), find("fpmodel.pt", "MODEL_PT")
PC_DIR = os.path.dirname(find("pc_smiles.npy", "PC_SMILES")); PP_DIR = os.path.dirname(find("pc_lsid.npy", "PC_POP"))
VAL_KEYS = os.path.join(os.path.dirname(MODEL_PT), "val_keys.csv")
OUT_DIR = os.environ.get("CASMI_OUT", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
CFG = dict(bin_width=0.02, max_mz=1300.0, min_rel_int=0.01, max_peaks=100, min_peaks=2,
           mass_tol_da=0.01, mass_tol_ppm=10.0, max_cands=4000, pc_cands=4000, topk=25)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(TRAIN, POOL_CSV, NP_PQ, MODEL_PT, dev)

In [ ]:
# ---- masses ----
EL = {"H":1.00782503207,"C":12.0,"N":14.0030740048,"O":15.99491461956,"F":18.99840322,"Na":22.9897692809,
      "P":30.97376163,"S":31.972071,"Cl":34.96885268,"K":38.96370668,"Br":78.9183371,"I":126.904473,
      "B":11.0093054,"Si":27.9769265325,"Se":79.9165213,"As":74.9215965,"Fe":55.9349375,"Mg":23.985041699,
      "Ca":39.96259098,"Zn":63.9291422,"Cu":62.9295975,"Li":7.016003437,"Pt":194.964791,"Sn":119.9022016,
      "Hg":201.970643,"Co":58.933195,"Mn":54.9380451,"Ni":57.9353429,"Al":26.98153863,"Ge":73.9211778,"Sb":120.903818}
_tok = re.compile(r"([A-Z][a-z]?)(\d*)")
def formula_mass(f):
    if not isinstance(f, str) or not f: return np.nan
    m = 0.0
    for el, n in _tok.findall(f):
        if el not in EL: return np.nan
        m += EL[el] * (int(n) if n else 1)
    return m

PROTON, E = 1.007276467, 0.000548580
H2O, NH4, NA, K, CL, CH2O2 = 18.0105646863, 18.0343740, 22.9897692809, 38.96370668, 34.96885268, 46.00547930326
# neutral M = precursor_mz - ADDUCT_DELTA
ADDUCT_DELTA = {
    "[M+H]+": PROTON, "[M+NH4]+": NH4 + 0 - E, "[M-H2O+H]+": PROTON - H2O, "[M-2H2O+H]+": PROTON - 2*H2O,
    "[M+Na]+": NA - E, "[M+K]+": K - E, "[M-H]-": -PROTON, "[M-H2O-H]-": -PROTON - H2O,
    "[M+CH2O2-H]-": CH2O2 - PROTON, "[M+Cl]-": CL + E,
}
def neutral_mass(df):
    d = df["adduct"].map(ADDUCT_DELTA)
    return (df["precursor_mz"] - d).to_numpy(float)

def mol_masses(test_df):
    d = test_df.assign(nm=neutral_mass(test_df))
    return d.groupby("molecule_id")["nm"].median()

In [ ]:
# ---- spectrum -> sparse vector ----
NB = int(CFG["max_mz"] / CFG["bin_width"]) + 1
def vectorize(mzs_list, ints_list, prec):
    rows, cols, vals = [], [], []
    keep = np.zeros(len(mzs_list), bool)
    for i, (mz, it, p) in enumerate(zip(mzs_list, ints_list, prec)):
        mz = np.asarray(mz, float); it = np.asarray(it, float)
        if len(mz) == 0: continue
        it = it / it.max()
        m = (mz <= p + 2.0) & (it >= CFG["min_rel_int"]) & (mz < CFG["max_mz"]) & (mz > 0)
        mz, it = mz[m], it[m]
        if len(mz) < CFG["min_peaks"]: continue
        if len(mz) > CFG["max_peaks"]:
            top = np.argsort(-it)[:CFG["max_peaks"]]; mz, it = mz[top], it[top]
        b = (mz / CFG["bin_width"]).astype(np.int64)
        v = np.sqrt(it)
        s = pd.Series(v).groupby(b).max()          # merge peaks sharing a bin
        v = s.to_numpy(); b = s.index.to_numpy()
        v = v / np.linalg.norm(v)
        rows.append(np.full(len(b), i)); cols.append(b); vals.append(v); keep[i] = True
    if not rows: return sp.csr_matrix((len(mzs_list), NB)), keep
    X = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(len(mzs_list), NB))
    return X, keep

In [ ]:
# shared featurisation (pasted into both notebooks)
import numpy as np, scipy.sparse as sp
FRAG_BINS, LOSS_BINS = 1000, 500
ADDUCTS = ["[M+H]+","[M+NH4]+","[M-H2O+H]+","[M-2H2O+H]+","[M+Na]+","[M+K]+","[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+Cl]-"]
N_SCALAR = len(ADDUCTS) + 3
IN_DIM = FRAG_BINS + LOSS_BINS + N_SCALAR
def featurize(mzs, ints, prec, adduct, ion):
    """-> CSR (n, IN_DIM); invalid spectra give empty rows."""
    r, c, v = [], [], []
    n = len(mzs)
    for i in range(n):
        mz = np.asarray(mzs[i], float); it = np.asarray(ints[i], float); p = float(prec[i])
        if len(mz) and it.max() > 0:
            it = it / it.max()
            m = (mz <= p + 2) & (it >= 0.005) & (mz > 0)
            mz, it = mz[m], np.sqrt(it[m])
            if len(mz) > 150:
                t = np.argsort(-it)[:150]; mz, it = mz[t], it[t]
            if len(mz):
                it = it / np.linalg.norm(it)
                fb = np.minimum(mz.astype(int), FRAG_BINS - 1)
                loss = p - mz; lm = (loss >= 0) & (loss < LOSS_BINS)
                r.append(np.full(len(fb), i)); c.append(fb); v.append(it)
                r.append(np.full(lm.sum(), i)); c.append(FRAG_BINS + loss[lm].astype(int)); v.append(it[lm])
        ai = ADDUCTS.index(adduct[i]) if adduct[i] in ADDUCTS else None
        base = FRAG_BINS + LOSS_BINS
        r.append(np.array([i, i, i])); c.append(np.array([base + len(ADDUCTS), base + len(ADDUCTS) + 1, base + len(ADDUCTS) + 2]))
        v.append(np.array([min(p, 1500) / 1000.0, 1.0 if ion[i] == "positive" else 0.0, 1.0 if ai is None else 0.0]))
        if ai is not None:
            r.append(np.array([i])); c.append(np.array([base + ai])); v.append(np.array([1.0]))
    X = sp.csr_matrix((np.concatenate(v), (np.concatenate(r), np.concatenate(c))), shape=(n, IN_DIM), dtype=np.float32)
    return X


In [ ]:
# ---- candidate pool ----
pool = pd.read_csv(POOL_CSV, usecols=["inchikey14","smiles","monoisotopic_mass","pop","np_coconut","np_lotus","np_npatlas"])
pool["np"] = ((pool.np_coconut + pool.np_lotus + pool.np_npatlas) > 0).astype(np.float32)
pool = pool.rename(columns={"monoisotopic_mass": "mass"})[["inchikey14","smiles","mass","pop","np"]]
npt = pd.read_parquet(NP_PQ, columns=["ik14","smiles","mono_mass","pubchem_substances","pubmed_articles"])
npt = npt[~npt.ik14.isin(set(pool.inchikey14))]
npt = pd.DataFrame({"inchikey14": npt.ik14, "smiles": npt.smiles, "mass": npt.mono_mass,
                    "pop": np.log1p(npt.pubchem_substances) + np.log1p(npt.pubmed_articles), "np": 1.0})
pool = pd.concat([pool, npt], ignore_index=True).dropna(subset=["mass","smiles"]).sort_values("mass").reset_index(drop=True)
pool_mass = pool.mass.to_numpy(); pool_key = pool.inchikey14.to_numpy()
print(len(pool), "candidate structures;", int(pool.np.sum()), "natural products")
lib_keys = set(pq.read_table(TRAIN, columns=["inchikey14"]).column(0).unique().to_pylist())
print(len(lib_keys), "structures with library spectra")
pc_mass = np.load(f"{PC_DIR}/pc_mass.npy", mmap_mode="r"); pc_off = np.load(f"{PC_DIR}/pc_off.npy", mmap_mode="r"); pc_buf = np.load(f"{PC_DIR}/pc_smiles.npy", mmap_mode="r")
pc_lsid = np.load(f"{PP_DIR}/pc_lsid.npy", mmap_mode="r"); pc_lpmid = np.load(f"{PP_DIR}/pc_lpmid.npy", mmap_mode="r")
pc_ik14 = np.load(f"{PP_DIR}/pc_ik14.npy", mmap_mode="r"); pc_np = np.load(f"{PP_DIR}/pc_np.npy", mmap_mode="r")
print("PubChem tier:", len(pc_mass), "rows")
pool_pop = pool["pop"].to_numpy(); pool_np = pool["np"].to_numpy(); pool_smi = pool.smiles.to_numpy()
SMI = {}

In [ ]:
# ---- fingerprint model ----
ck = torch.load(MODEL_PT, map_location=dev)
class Net(nn.Module):
    def __init__(s, d_in, h, out):
        super().__init__(); s.f = nn.Sequential(nn.Linear(d_in,h), nn.ReLU(), nn.Dropout(0), nn.Linear(h,h), nn.ReLU(), nn.Dropout(0), nn.Linear(h,out))
    def forward(s, x): return s.f(x)
net = Net(ck["in_dim"], ck["hidden"], ck["fp_bits"]).to(dev); net.load_state_dict(ck["state"]); net.eval()
_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=ck["fp_bits"])
_fpc = {}
def cand_fp(key, smi):
    f = _fpc.get(key)
    if f is None:
        m = Chem.MolFromSmiles(smi)
        f = np.zeros(ck["fp_bits"], np.float32) if m is None else _gen.GetFingerprintAsNumPy(m).astype(np.float32)
        _fpc[key] = f
    return f
def mol_probs(g):
    X = featurize(g.ms2_mzs.tolist(), g.ms2_normalized_intensities.tolist(), g.precursor_mz.to_numpy(), g.adduct.tolist(), g.ionization_mode.tolist())
    with torch.no_grad(): P = torch.sigmoid(net(torch.from_numpy(X.toarray()).to(dev))).cpu().numpy()
    return P.mean(0)

In [ ]:
def load_library_vectors(keys, exclude_rows=frozenset(), exclude_keys=frozenset()):
    keys = pa.array(sorted(set(keys) - set(exclude_keys)))
    pf = pq.ParquetFile(TRAIN); cols = ["inchikey14","ms2_mzs","ms2_normalized_intensities","precursor_mz"]
    mats, metas, off = [], [], 0
    for b in pf.iter_batches(batch_size=50000, columns=cols):
        t = pa.Table.from_batches([b]); t = t.append_column("spectrum_id", pa.array(np.arange(off, off + t.num_rows))); off += t.num_rows
        t = t.filter(pc.is_in(t["inchikey14"], value_set=keys))
        if t.num_rows == 0: continue
        df = t.to_pandas()
        if exclude_rows: df = df[~df.spectrum_id.isin(exclude_rows)]
        X, keep = vectorize(df.ms2_mzs, df.ms2_normalized_intensities, df.precursor_mz.to_numpy(float))
        mats.append(X[np.flatnonzero(keep)]); metas.append(df.loc[keep, ["spectrum_id","inchikey14"]])
    X = sp.vstack(mats).tocsr() if mats else sp.csr_matrix((0, NB))
    meta = pd.concat(metas, ignore_index=True) if metas else pd.DataFrame(columns=["spectrum_id","inchikey14"])
    return X, meta

def candidates(mass, excl_pool=frozenset()):
    """Pool candidates (mass window) + PubChem-tier candidates not already in the pool."""
    tol = max(CFG["mass_tol_da"], mass * CFG["mass_tol_ppm"] * 1e-6)
    lo, hi = np.searchsorted(pool_mass, [mass - tol, mass + tol]); idx = np.arange(lo, hi)
    if excl_pool: idx = idx[~np.isin(pool_key[idx], list(excl_pool))]
    if len(idx) > CFG["max_cands"]: idx = idx[np.argsort(-pool_pop[idx])[:CFG["max_cands"]]]
    keys = list(pool_key[idx]); smis = list(pool_smi[idx]); pop = list(pool_pop[idx]); npf = list(pool_np[idx])
    a = int(np.searchsorted(pc_mass, mass - tol, "left")); b = int(np.searchsorted(pc_mass, mass + tol, "right"))
    if b > a:
        p = pc_lsid[a:b].astype(np.float32) + pc_lpmid[a:b].astype(np.float32)
        k14 = np.char.decode(np.asarray(pc_ik14[a:b]), "ascii")
        have = set(keys) | (set(pool_key[lo:hi]) if not excl_pool else set(pool_key[lo:hi]) - set(excl_pool))
        order = np.argsort(-p); seen = set(); take = []
        for j in order:
            k = k14[j]
            if k in seen or k in have or not k: continue
            seen.add(k); take.append(j)
            if len(take) >= CFG["pc_cands"]: break
        if take:
            off = np.asarray(pc_off[a:b + 1]).astype(np.int64)
            for j in take:
                keys.append(k14[j]); smis.append(bytes(pc_buf[off[j]:off[j + 1]]).decode("ascii")); pop.append(float(p[j])); npf.append(float(pc_np[a + j] > 0))
    for k, s_ in zip(keys, smis): SMI.setdefault(k, s_)
    return dict(keys=np.array(keys), smiles=smis, pop=np.array(pop, np.float32), np=np.array(npf, np.float32))

def components(test_df, exclude_rows=frozenset(), exclude_keys=frozenset(), excl_pool=frozenset()):
    """Per molecule: candidate idx + lib / fp / pop / np component arrays."""
    mm = mol_masses(test_df)
    cand = {m: candidates(x, excl_pool) for m, x in mm.items() if np.isfinite(x)}
    allkeys = {k for v in cand.values() for k in v["keys"]} & lib_keys
    t0 = time.time(); LX, Lmeta = load_library_vectors(allkeys, exclude_rows, exclude_keys)
    print(f"library: {LX.shape[0]} spectra / {len(allkeys)} structures ({time.time()-t0:.0f}s)")
    by_key = Lmeta.groupby("inchikey14").indices; out = {}
    for n, (mol, g) in enumerate(test_df.groupby("molecule_id")):
        cd = cand.get(mol)
        if cd is None or len(cd["keys"]) == 0: out[mol] = None; continue
        keys = cd["keys"]
        p = mol_probs(g)
        F = np.stack([cand_fp(k, s) for k, s in zip(keys, cd["smiles"])])
        inter = F @ p; fp = inter / (p.sum() + F.sum(1) - inter + 1e-9)
        lib = np.zeros(len(keys), np.float32)
        have = [j for j, k in enumerate(keys) if k in by_key]
        if have:
            QX, keep = vectorize(g.ms2_mzs, g.ms2_normalized_intensities, g.precursor_mz.to_numpy(float)); QX = QX[np.flatnonzero(keep)]
            if QX.shape[0]:
                rows = np.concatenate([by_key[keys[j]] for j in have]); own = np.concatenate([[q]*len(by_key[keys[j]]) for q, j in enumerate(have)])
                S = (QX @ LX[rows].T).toarray()
                for q, j in enumerate(have): lib[j] = S[:, own == q].max(axis=1).mean()
        out[mol] = dict(keys=keys, lib=lib, fp=fp.astype(np.float32), pop=cd["pop"], np=cd["np"])
    return out

def blend(c, w):
    fz = (c["fp"] - c["fp"].mean()) / (c["fp"].std() + 1e-6)
    pz = (c["pop"] - c["pop"].mean()) / (c["pop"].std() + 1e-6)
    t = w.get("t", 0.0); le = np.maximum(c["lib"] - t, 0) / (1 - t)      # only strong library matches count
    return w["lib"] * le + w["fp"] * fz + w["pop"] * pz + w["np"] * c["np"]

def rank_keys(comps, w, k=25):
    out = {}
    for mol, c in comps.items():
        if c is None: out[mol] = []; continue
        o = np.argsort(-blend(c, w))[:k]; out[mol] = [c["keys"][j] for j in o]
    return out

def mrr(ranked, truth, k=25):
    rr = [(1.0 / (ranked[m].index(t) + 1)) if t in ranked.get(m, [])[:k] else 0.0 for m, t in truth.items()]
    return float(np.mean(rr)) if rr else float("nan")

## Validation mix

* **class-2 proxy**: held-out structures from the FP-model's validation split; *all* their spectra are removed from the library (the model never saw them either). Candidate pool still contains them.
* **class-1 proxy**: random other structures, half of their spectra held out and removed from the library.

Weights are tuned on a mix (30% class-1 / 70% class-2, matching the ~0.14 public score of library-only retrieval).

In [ ]:
N_EACH = int(os.environ.get("N_VAL", 200))
meta = pq.read_table(TRAIN, columns=["inchikey14","adduct"]).to_pandas(); meta["row"] = np.arange(len(meta))
meta = meta[meta.adduct.isin(ADDUCT_DELTA)]
rng = np.random.RandomState(1)
vk = pd.read_csv(VAL_KEYS).inchikey14
v2 = rng.choice(vk[vk.isin(set(meta.inchikey14)) & vk.isin(set(pool_key))].to_numpy(), N_EACH, replace=False)  # (class-2 proxies are removed from the pool during validation, so only the PubChem tier can supply them)
cnt = meta[~meta.inchikey14.isin(set(vk))].groupby("inchikey14").size(); m1 = cnt[cnt >= 2].index.to_numpy()
v1 = rng.choice(m1, N_EACH, replace=False)
rows2 = meta[meta.inchikey14.isin(set(v2))]
rows1 = meta[meta.inchikey14.isin(set(v1))].groupby("inchikey14", group_keys=False).apply(lambda g: g.sample(frac=0.5, random_state=0) if len(g) > 2 else g.iloc[:1])
use = pd.concat([rows2, rows1]); want = set(use.row)
parts, off = [], 0; sorted_want = np.array(sorted(want))
for b in pq.ParquetFile(TRAIN).iter_batches(batch_size=50000, columns=["inchikey14","ms2_mzs","ms2_normalized_intensities","adduct","ionization_mode","precursor_mz"]):
    lo, hi = np.searchsorted(sorted_want, [off, off + b.num_rows])
    if hi > lo:
        d = pa.Table.from_batches([b]).take(pa.array(sorted_want[lo:hi] - off)).to_pandas(); d["row"] = sorted_want[lo:hi]; parts.append(d)
    off += b.num_rows
val_df = pd.concat(parts, ignore_index=True); val_df["molecule_id"] = val_df.inchikey14
truth = {k: k for k in val_df.inchikey14.unique()}
cls = {k: (2 if k in set(v2) else 1) for k in truth}
t0 = time.time()
vcomp = components(val_df, exclude_rows=set(rows1.row), exclude_keys=set(v2), excl_pool=set(v2))
print(f"components done {time.time()-t0:.0f}s")

In [ ]:
def score_mix(w, a=0.3):
    r = rank_keys(vcomp, w)
    t1 = {m: t for m, t in truth.items() if cls[m] == 1}; t2 = {m: t for m, t in truth.items() if cls[m] == 2}
    s1, s2 = mrr(r, t1), mrr(r, t2); return a * s1 + (1 - a) * s2, s1, s2
best = None
for t, wl, wf, wp, wn in itertools.product([0.5, 0.7, 0.85], [0, 5, 20, 50], [0.5, 1, 2, 4], [0, 0.25, 0.5], [0, 0.5, 1]):
    w = dict(t=t, lib=wl, fp=wf, pop=wp, np=wn); s = score_mix(w)
    if best is None or s[0] > best[0][0]: best = (s, w)
print("best mix %.4f (class1 %.4f, class2 %.4f)" % best[0], best[1])
for name, w in [("lib only", dict(lib=1, fp=0, pop=0, np=0)), ("fp only", dict(lib=0, fp=1, pop=0, np=0)), ("pop only", dict(lib=0, fp=0, pop=1, np=0))]:
    print(name, "mix %.4f c1 %.4f c2 %.4f" % score_mix(w))
W = best[1]

## Predict test and write `submission.csv`

In [ ]:
test = pq.read_table(f"{DATA_DIR}/test.parquet").to_pandas()
tcomp = components(test)
ranked = rank_keys(tcomp, W)
smi = SMI
rows = [(m, ";".join(smi[k] for k in ranked.get(m, [])[:25]) or "CCO") for m in test.molecule_id.unique()]
sub = pd.DataFrame(rows, columns=["molecule_id", "smiles"]); sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)
print((sub.smiles == "CCO").sum(), "molecules without candidates"); sub.head()

In [ ]:
ss = pd.read_csv(f"{DATA_DIR}/sample_submission.csv"); chk = pd.read_csv(f"{OUT_DIR}/submission.csv")
assert list(chk.columns) == ["molecule_id", "smiles"] and chk.molecule_id.is_unique and chk.notna().all().all()
assert set(chk.molecule_id) == set(ss.molecule_id) and chk.smiles.str.count(";").add(1).max() <= 25
print("submission OK", chk.shape)